# Letter-count eval: failure-mode breakdown

Analyzes the annotated inference JSONLs saved by the `count_letters` eval task
(`src/oumi/evaluation/registry/count_letters_task.py`) and breaks down every
*invalid* response (no parseable `\boxed{N}` answer) into failure modes:

- **multiple boxes** — more than one `\boxed{}` in the response (the extractor requires exactly one)
- **truncated** — the response hit the `max_new_tokens` cap before finishing
- **natural end, no box** — the model finished (EOS) without ever emitting `\boxed{}`, further split by
  whether the correct count is stated in prose ("The letter 'g' appears 1 time")

Run the evals first, e.g.:
```bash
VLLM_WORKER_MULTIPROC_METHOD=spawn oumi evaluate -c output/letter_counting.grpo_verl_v2/eval_trained.yaml
VLLM_WORKER_MULTIPROC_METHOD=spawn oumi evaluate -c output/letter_counting.grpo_verl_v2/eval_base_gemma4_e2b_it.yaml
```

In [1]:
import json
import re
from pathlib import Path

import pandas as pd
from transformers import AutoTokenizer

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
EVAL_ROOT = REPO_ROOT / "output/letter_counting.grpo_verl_v2/evaluation"

# Model runs to compare: label -> eval output dir (the newest
# count_letters_inference_*.jsonl in each dir is used).
RUNS = {
    "base gemma-4-E2B-it": EVAL_ROOT / "base_gemma4_e2b_it",
    "trained (GRPO)": EVAL_ROOT / "trained",
}

# Tokenizer used to detect truncation (any tokenizer of the same family works).
TOKENIZER_PATH = REPO_ROOT / "output/letter_counting.grpo_verl_v2/final_model"
# `max_new_tokens` used at eval time; responses within a few tokens of the cap
# are counted as truncated.
MAX_NEW_TOKENS = 2048
TRUNCATION_THRESHOLD = MAX_NEW_TOKENS - 8

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH)

In [2]:
BOX_RE = re.compile(r"\\boxed\{([-+]?\d+)\}")

# Number words the model uses when answering in prose.
_NUMBER_WORDS = {
    "zero": 0,
    "once": 1,
    "one": 1,
    "twice": 2,
    "two": 2,
    "three": 3,
    "thrice": 3,
    "four": 4,
    "five": 5,
    "six": 6,
    "seven": 7,
    "eight": 8,
    "nine": 9,
    "ten": 10,
}
# Phrases like "appears once", "there are two", "3 times", "count = 1".
_PROSE_COUNT_RE = re.compile(
    r"appears\s+(\w+)|there (?:is|are)\s+(\w+)|(\w+)\s+times?\b"
    r"|count\s*[=:]?\s*(\d+)|frequency[^0-9]*(\d+)"
)


def load_records(eval_dir: Path) -> list[dict]:
    """Loads the newest count_letters_inference_*.jsonl in `eval_dir`."""
    files = sorted(eval_dir.glob("count_letters_inference_*.jsonl"))
    if not files:
        raise FileNotFoundError(f"No count_letters_inference_*.jsonl in {eval_dir}")
    path = files[-1]
    print(f"Loading {path}")
    with path.open() as f:
        return [json.loads(line) for line in f]


def extract_prose_counts(response_tail: str) -> set[int]:
    """Returns the counts stated in prose in the response tail (heuristic)."""
    counts: set[int] = set()
    for match in _PROSE_COUNT_RE.finditer(response_tail.lower()):
        for group in match.groups():
            if group is None:
                continue
            if group.isdigit():
                counts.add(int(group))
            elif group in _NUMBER_WORDS:
                counts.add(_NUMBER_WORDS[group])
    return counts


def classify_invalid(record: dict) -> str:
    """Classifies one invalid (prediction is None) record into a failure mode."""
    response = record["messages"][-1]["content"]
    if len(BOX_RE.findall(response)) > 1:
        return "multiple boxes"
    n_tokens = len(tokenizer(response)["input_ids"])
    if n_tokens >= TRUNCATION_THRESHOLD:
        return "truncated at cap"
    ground_truth = record["metadata"]["letter_count_integer"]
    prose_counts = extract_prose_counts(response[-300:])
    if not prose_counts:
        return "natural end / prose unparseable"
    if prose_counts == {ground_truth}:
        return "natural end / prose count correct"
    return "natural end / prose count wrong or ambiguous"

In [3]:
ROW_ORDER = [
    "total records",
    "valid (boxed) answers",
    "invalid total",
    "multiple boxes",
    "truncated at cap",
    "natural end, no box",
    "— of which: correct count in prose",
    "— of which: wrong/ambiguous count in prose",
    "— of which: no count parseable from prose",
]

MODE_TO_ROW = {
    "multiple boxes": "multiple boxes",
    "truncated at cap": "truncated at cap",
    "natural end / prose count correct": "— of which: correct count in prose",
    "natural end / prose count wrong or ambiguous": "— of which: wrong/ambiguous count in prose",
    "natural end / prose unparseable": "— of which: no count parseable from prose",
}

all_records: dict[str, list[dict]] = {}
breakdown: dict[str, dict[str, int]] = {}

for label, eval_dir in RUNS.items():
    records = load_records(eval_dir)
    all_records[label] = records
    counts = dict.fromkeys(ROW_ORDER, 0)
    counts["total records"] = len(records)
    for record in records:
        if record["metadata"]["prediction"] is not None:
            counts["valid (boxed) answers"] += 1
            continue
        counts["invalid total"] += 1
        counts[MODE_TO_ROW[classify_invalid(record)]] += 1
    counts["natural end, no box"] = (
        counts["— of which: correct count in prose"]
        + counts["— of which: wrong/ambiguous count in prose"]
        + counts["— of which: no count parseable from prose"]
    )
    breakdown[label] = counts

table = pd.DataFrame(breakdown).loc[ROW_ORDER]
table.index.name = "failure mode"
table

Loading /workspace/persist/shanghong/oumi/output/letter_counting.grpo_verl_v2/evaluation/base_gemma4_e2b_it/count_letters_inference_20260824_191529.jsonl
Loading /workspace/persist/shanghong/oumi/output/letter_counting.grpo_verl_v2/evaluation/trained/count_letters_inference_20260824_191444.jsonl


,base gemma-4-E2B-it,trained (GRPO)
failure mode,,
total records,1000,1000
valid (boxed) answers,384,649
invalid total,616,351
multiple boxes,0,0
truncated at cap,9,6
"natural end, no box",607,345
— of which: correct count in prose,452,267
— of which: wrong/ambiguous count in prose,27,16
— of which: no count parseable from prose,128,62


## Inspect individual failures

Change `RUN_LABEL` / `MODE` / `N_EXAMPLES` to browse specific failure modes.

In [4]:
RUN_LABEL = "trained (GRPO)"
MODE = "natural end / prose count correct"  # see classify_invalid() for modes
N_EXAMPLES = 3

shown = 0
for record in all_records[RUN_LABEL]:
    if record["metadata"]["prediction"] is not None:
        continue
    if classify_invalid(record) != MODE:
        continue
    meta = record["metadata"]
    print(
        f"word={meta['word']!r} letter={meta['letter']!r} "
        f"ground_truth={meta['letter_count_integer']}"
    )
    print("response tail:", repr(record["messages"][-1]["content"][-300:]))
    print("-" * 100)
    shown += 1
    if shown >= N_EXAMPLES:
        break

word='legalize' letter='g' ground_truth=1
response tail: "'.\nWe need to count the number of times the letter 'g' appears in this word.\n\nLet's examine the letters in 'legalize':\nl - e - g - a - l - i - z - e\n\nCounting the occurrences of 'g':\n1. l\n2. e\n3. **g** (1st occurrence)\n4. a\n5. l\n6. i\n7. z\n8. e\n\nThe letter 'g' appears 1 time in the word 'legalize'.\n\n"
----------------------------------------------------------------------------------------------------
word='immund' letter='n' ground_truth=1
response tail: "The word is 'immund'.\nWe need to count the occurrences of the letter 'n' in this word.\n\nLet's examine the letters in 'immund':\ni - 1st letter\nm - 2nd letter\nm - 3rd letter\nu - 4th letter\nn - 5th letter\nd - 6th letter\n\nThe letter 'n' appears once in the word 'immund'.\n\n"
----------------------------------------------------------------------------------------------------
word='hongkong' letter='n' ground_truth=2
response tail: "The word is 'hongkong'